# 02 - Parsing with Drain3

For each dataset (HDFS, BGL, OpenSSH, Apache) we:

1. read the raw `_2k.log` file and extract the message part of each line (checked against the `Content` column of the structured CSV)
2. parse the messages with Drain3 (`sim_th=0.4`, `depth=4`, masking IPs, block IDs, hex values and numbers)
3. compare the number of Drain templates with the true number of `EventId`s
4. compute **grouping accuracy**: a line is correct only if the group Drain put it in contains exactly the same lines as its true `EventId` group
5. show 5 example lines next to the template Drain produced

All parsing code lives in `src/parsing/drain_parser.py` so the live system can reuse it.

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.parsing.drain_parser import MASKING_INSTRUCTIONS, grouping_accuracy, parse_log_file

pd.set_option("display.max_colwidth", 120)

RAW = ROOT / "data" / "raw"
DOCS = ROOT / "docs"
DATASETS = ["HDFS", "BGL", "OpenSSH", "Apache"]
SIM_TH, DEPTH = 0.4, 4

pd.DataFrame(
    [(mi.mask_with, mi.pattern) for mi in MASKING_INSTRUCTIONS],
    columns=["Mask", "Regex (applied in this order)"],
)

,Mask,Regex (applied in this order)
0,BLK,blk_-?\d+
1,IP,"((?<=[^A-Za-z0-9])|^)(\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3})(:\d+)?((?=[^A-Za-z0-9])|$)"
2,HEX,((?<=[^A-Za-z0-9])|^)(0x[a-fA-F0-9]+)((?=[^A-Za-z0-9])|$)
3,NUM,((?<=[^A-Za-z0-9])|^)([\-\+]?\d+)((?=[^A-Za-z0-9])|$)


In [2]:
def evaluate(name):
    parsed = parse_log_file(RAW / name / f"{name}_2k.log", name, sim_th=SIM_TH, depth=DEPTH)
    truth = pd.read_csv(RAW / name / f"{name}_2k.log_structured.csv", dtype=str)
    truth["LineId"] = truth["LineId"].astype(int)
    df = parsed.merge(truth[["LineId", "Content", "EventId"]], on="LineId", suffixes=("", "_true"))

    content_match = (df["Content"] == df["Content_true"]).mean() * 100
    drain_templates = df["ClusterId"].nunique()
    true_templates = df["EventId"].nunique()
    accuracy = grouping_accuracy(df["EventId"], df["ClusterId"]) * 100

    print(f"Lines parsed:                 {len(df)}")
    print(f"Extracted message == Content: {content_match:.2f}%")
    print(f"Drain templates:              {drain_templates}")
    print(f"True templates (EventId):     {true_templates}")
    print(f"Grouping accuracy:            {accuracy:.2f}%")

    # One example from each of the 5 largest Drain clusters, so the examples show different templates
    top_clusters = df["ClusterId"].value_counts().index[:5]
    examples = df[df["ClusterId"].isin(top_clusters)].drop_duplicates("ClusterId")
    display(examples[["Raw", "Template"]].reset_index(drop=True))

    return df, {
        "dataset": name,
        "drain_templates": drain_templates,
        "true_templates": true_templates,
        "grouping_accuracy_pct": round(accuracy, 2),
    }


results, summary = {}, []

## HDFS

In [3]:
results["HDFS"], row = evaluate("HDFS")
summary.append(row)

Lines parsed:                 2000
Extracted message == Content: 100.00%
Drain templates:              16
True templates (EventId):     14
Grouping accuracy:            99.75%


,Raw,Template
0,081109 203615 148 INFO dfs.DataNode$PacketResponder: PacketResponder 1 for block blk_38865049064139660 terminating,PacketResponder <NUM> for block <BLK> terminating
1,081109 204005 35 INFO dfs.FSNamesystem: BLOCK* NameSystem.addStoredBlock: blockMap updated: 10.251.73.220:50010 is a...,BLOCK* NameSystem.addStoredBlock: blockMap updated: <IP> is added to <BLK> size <NUM>
2,081109 204655 556 INFO dfs.DataNode$PacketResponder: Received block blk_3587508140051953248 of size 67108864 from /1...,Received block <BLK> of size <NUM> from /<IP>
3,081109 204815 653 INFO dfs.DataNode$DataXceiver: Receiving block blk_5792489080791696128 src: /10.251.30.6:33145 des...,Receiving block <BLK> src: /<IP> dest: /<IP>
4,081109 213837 19 INFO dfs.FSDataset: Deleting block blk_1781953582842324563 file /mnt/hadoop/dfs/data/current/subdir...,Deleting block <BLK> file <*>


## BGL

In [4]:
results["BGL"], row = evaluate("BGL")
summary.append(row)

Lines parsed:                 2000
Extracted message == Content: 100.00%
Drain templates:              105
True templates (EventId):     120
Grouping accuracy:            96.85%


,Raw,Template
0,- 1117842440 2005.06.03 R23-M0-NE-C:J05-U01 2005-06-03-16.47.20.730545 R23-M0-NE-C:J05-U01 RAS KERNEL INFO 63543 dou...,<NUM> double-hummer alignment exceptions
1,"- 1117848119 2005.06.03 R16-M1-N2-C:J17-U01 2005-06-03-18.21.59.871925 R16-M1-N2-C:J17-U01 RAS KERNEL INFO CE sym 2,...","CE sym <NUM>, at <HEX>, mask <HEX>"
2,- 1117955341 2005.06.05 R25-M0-N7-C:J02-U01 2005-06-05-00.09.01.903373 R25-M0-N7-C:J02-U01 RAS KERNEL INFO generatin...,generating core.<NUM>
3,- 1121900469 2005.07.20 R33-M1-N3-C:J02-U01 2005-07-20-16.01.09.861175 R33-M1-N3-C:J02-U01 RAS KERNEL INFO 8 floatin...,<NUM> floating point alignment exceptions
4,- 1125082053 2005.08.26 R62-M1-N3-C:J04-U01 2005-08-26-11.47.33.903841 R62-M1-N3-C:J04-U01 RAS KERNEL INFO iar 003a9...,iar <*> dear <*>


## OpenSSH

In [5]:
results["OpenSSH"], row = evaluate("OpenSSH")
summary.append(row)

Lines parsed:                 2000
Extracted message == Content: 100.00%
Drain templates:              23
True templates (EventId):     27
Grouping accuracy:            71.80%


,Raw,Template
0,Dec 10 06:55:46 LabSZ sshd[24200]: pam_unix(sshd:auth): check pass; user unknown,pam_unix(sshd:auth): check pass; user unknown
1,Dec 10 06:55:48 LabSZ sshd[24200]: Failed password for invalid user webmaster from 173.234.31.186 port 38926 ssh2,Failed <*> for invalid user <*> from <IP> port <NUM> ssh2
2,Dec 10 07:07:45 LabSZ sshd[24206]: Received disconnect from 52.80.34.196: 11: Bye Bye [preauth],Received disconnect from <IP>: <NUM>: <*> <*> <*>
3,Dec 10 07:13:31 LabSZ sshd[24227]: pam_unix(sshd:auth): authentication failure; logname= uid=0 euid=0 tty=ssh ruser=...,pam_unix(sshd:auth): authentication failure; logname= uid=<NUM> euid=<NUM> tty=ssh ruser= <*> <*>
4,Dec 10 07:13:43 LabSZ sshd[24227]: Failed password for root from 5.36.59.76 port 42393 ssh2,Failed password for <*> from <IP> port <NUM> ssh2


## Apache

In [6]:
results["Apache"], row = evaluate("Apache")
summary.append(row)

Lines parsed:                 2000
Extracted message == Content: 100.00%
Drain templates:              6
True templates (EventId):     6
Grouping accuracy:            100.00%


,Raw,Template
0,[Sun Dec 04 04:47:44 2005] [notice] workerEnv.init() ok /etc/httpd/conf/workers2.properties,workerEnv.init() ok /etc/httpd/conf/workers2.properties
1,[Sun Dec 04 04:47:44 2005] [error] mod_jk child workerEnv in error state 6,mod_jk child workerEnv in error state <NUM>
2,[Sun Dec 04 04:51:08 2005] [notice] jk2_init() Found child 6725 in scoreboard slot 10,jk2_init() Found child <NUM> in scoreboard slot <NUM>
3,[Sun Dec 04 05:15:09 2005] [error] [client 222.166.160.184] Directory index forbidden by rule: /var/www/html/,[client <IP>] Directory index forbidden by rule: /var/www/html/
4,[Sun Dec 04 17:43:08 2005] [error] jk2_init() Can't find child 1566 in scoreboard,jk2_init() Can't find child <NUM> in scoreboard


## Where Drain gets it wrong

For the dataset with the lowest accuracy, list the true `EventId` groups that Drain did not reproduce exactly, and which Drain clusters their lines ended up in.

In [7]:
worst = min(summary, key=lambda r: r["grouping_accuracy_pct"])["dataset"]
df = results[worst]
print(f"Lowest accuracy: {worst}")

pairs = df.groupby(["EventId", "ClusterId"]).size().rename("Lines").reset_index()
cluster_event_count = pairs.groupby("ClusterId")["EventId"].transform("nunique")
event_cluster_count = pairs.groupby("EventId")["ClusterId"].transform("nunique")
wrong = pairs[(cluster_event_count > 1) | (event_cluster_count > 1)].copy()
wrong["Template"] = wrong["ClusterId"].map(df.drop_duplicates("ClusterId").set_index("ClusterId")["Template"])
wrong.sort_values(["ClusterId", "EventId"]).reset_index(drop=True)

Lowest accuracy: OpenSSH


,EventId,ClusterId,Lines,Template
0,E10,6,135,Failed <*> for invalid user <*> from <IP> port <NUM> ssh2
1,E8,6,4,Failed <*> for invalid user <*> from <IP> port <NUM> ssh2
2,E24,8,413,Received disconnect from <IP>: <NUM>: <*> <*> <*>
3,E26,8,1,Received disconnect from <IP>: <NUM>: <*> <*> <*>
4,E4,12,1,Disconnecting: Too many authentication failures for <*> [preauth]
5,E5,12,2,Disconnecting: Too many authentication failures for <*> [preauth]
6,E15,18,2,PAM <NUM> more authentication <*> logname= uid=<NUM> euid=<NUM> tty=ssh ruser= rhost=<IP>
7,E16,18,6,PAM <NUM> more authentication <*> logname= uid=<NUM> euid=<NUM> tty=ssh ruser= rhost=<IP>


A Drain cluster listed against several `EventId`s means Drain **merged** templates that should be separate; an `EventId` listed against several clusters means Drain **split** one template into several.

## Summary

In [8]:
summary_df = pd.DataFrame(summary)

DOCS.mkdir(exist_ok=True)
summary_df.to_csv(DOCS / "parsing_results.csv", index=False)
summary_df

,dataset,drain_templates,true_templates,grouping_accuracy_pct
0,HDFS,16,14,99.75
1,BGL,105,120,96.85
2,OpenSSH,23,27,71.80
3,Apache,6,6,100.00
